---
title: "Persistence, Replay, and Memory"
draft: true
categories: [agents, workflows, langgraph]
---


Checkpointing turns an in-process invocation into a resumable thread. It also creates storage, migration, privacy, and replay responsibilities. The local demonstration uses SQLite because reopening the file provides a real process-boundary test.

## Restart from a durable interrupt

The database lives under the repository's gitignored `.tmp` directory. Strict msgpack mode narrows deserialization, and a stable `thread_id` identifies the paused execution.


In [1]:
import os
from pathlib import Path
os.environ["LANGGRAPH_STRICT_MSGPACK"] = "true"

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command
from evidence_brief.fixtures import request_for
from evidence_brief.workflow import build_evidence_brief_graph, make_context

tmp_dir = Path.cwd() / ".tmp" / "evidence-brief"
tmp_dir.mkdir(parents=True, exist_ok=True)
database = tmp_dir / "chapter-07.sqlite"
database.unlink(missing_ok=True)
config = {"configurable": {"thread_id": "durable-review"}}
context = make_context()

with SqliteSaver.from_conn_string(str(database)) as saver:
    graph = build_evidence_brief_graph(checkpointer=saver)
    paused = graph.invoke(
        {"request": request_for("conflict-01").model_dump(), "events": [], "branch_results": []},
        config=config, context=context, version="v2",
    )
    print("paused at", graph.get_state(config).next)

with SqliteSaver.from_conn_string(str(database)) as saver:
    restarted = build_evidence_brief_graph(checkpointer=saver)
    completed = restarted.invoke(
        Command(resume={"action": "approve", "reason": "restart verified"}),
        config=config, context=context, version="v2",
    )
    history = list(restarted.get_state_history(config))

print({"status": completed.value["status"], "checkpoints": len(history), "effects": context.controller.effects})
assert completed.value["status"] == "complete"
assert context.controller.effects.count("collect:security") == 1
assert context.controller.effects.count("export:artifact") == 1


paused at ('review',)


{'status': 'complete', 'checkpoints': 11, 'effects': ['collect:security', 'collect:performance', 'collect:operations', 'export:artifact']}


Reconstructing both the saver and graph did not repeat collection. The checkpoint owns workflow progress; the idempotent effect ledger independently proves which external actions occurred.

## Replay, fork, subgraphs, and stores

Replay resumes from an existing checkpoint. A fork first creates a new checkpoint with changed state. Long-term memory is separate again: it is namespaced in a store rather than embedded in one thread's checkpoint.


In [2]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore

with SqliteSaver.from_conn_string(str(database)) as saver:
    graph = build_evidence_brief_graph(checkpointer=saver)
    before_draft = next(snapshot for snapshot in graph.get_state_history(config) if snapshot.next == ("draft",))
    fork_config = graph.update_state(before_draft.config, {"contradictions": []})
    forked = graph.invoke(None, fork_config, context=context, version="v2")
    assert forked.interrupts

class QualityState(TypedDict):
    claims: list[dict]
    quality: str

def assess_quality(state: QualityState):
    complete = all(claim.get("source_id") and claim.get("passage_id") for claim in state["claims"])
    return {"quality": "pass" if complete else "fail"}

quality_builder = StateGraph(QualityState)
quality_builder.add_node("assess_quality", assess_quality)
quality_builder.add_edge(START, "assess_quality")
quality_builder.add_edge("assess_quality", END)
quality_subgraph = quality_builder.compile()  # per-invocation; inherits a parent saver when nested
quality_result = quality_subgraph.invoke({"claims": completed.value["claims"], "quality": "unknown"})

store = InMemoryStore()
store.put(("reviewer-1", "preferences"), "source-policy", {"value": "prefer independent audits"})
preference = store.get(("reviewer-1", "preferences"), "source-policy").value

checkpoint_keys = sorted(completed.value.keys())
migrated = {**completed.value, "schema_version": 2}
print({
    "fork_paused_again": bool(forked.interrupts),
    "subgraph_quality": quality_result["quality"],
    "memory": preference,
    "additive_migration": migrated["schema_version"],
    "checkpoint_fields": len(checkpoint_keys),
})
assert quality_result["quality"] == "pass"
assert "schema_version" not in completed.value and migrated["schema_version"] == 2


{'fork_paused_again': True, 'subgraph_quality': 'pass', 'memory': {'value': 'prefer independent audits'}, 'additive_migration': 2, 'checkpoint_fields': 11}


The original history remains intact after the fork. An additive state field is compatible with old checkpoints; renaming a field would discard its saved value and therefore requires an explicit migration. Per-invocation quality-review subgraphs should inherit the parent checkpointer, while reviewer preferences belong in a separately retained store.
